# 01_Toolkit_Playground

Consolidated from the original workshop notebooks without changing any original code cells. New markdown/cells were added only to match the requested structure.

---

## Setup


## Step 1: Install Dependencies


In [ ]:
%pip install -q -r ../requirements.txt


## Step 2: Load Shared Notebook Utilities


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/ESMT_Workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Clear stale workshop modules to avoid import cache issues after code updates.
for module_name in [name for name in list(sys.modules) if name.startswith('esmt_workshop')]:
    del sys.modules[module_name]

from esmt_workshop.api_client import get_workshop_model_catalog
from esmt_workshop.evaluation import evaluate_predictions
from esmt_workshop.student_api import call_llm, process_batch_addresses, process_single_address
from esmt_workshop.student_utils import parse_llm_json, parse_llm_structured_address

print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Generate Workshop CSV Files (`dev/test`)


In [ ]:
import subprocess

# Build deterministic dev/test splits for workshop tasks.
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'scripts/prepare_workshop_data.py'),
    '--input-xlsx',
    str(PROJECT_ROOT / 'data/input/reference_address_cropped_Unstructured_col_100.xlsx'),
    '--output-dir',
    str(PROJECT_ROOT / 'data/workshop'),
    '--dev-size',
    '0.7',
]

subprocess.run(cmd, check=True)


## Step 4: Inspect Dataset Artifacts


In [ ]:
# Load generated split files for quick sanity checks.
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
test_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/test_labeled.csv', dtype=str).fillna('')
unlabeled_test_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/test_unlabeled.csv', dtype=str).fillna('')

print('dev/test/unlabeled_test sizes:', len(dev_df), len(test_df), len(unlabeled_test_df))
dev_df.head(5)


## Step 5: Student Identity, Model Picker, and Shared Generation Controls


In [ ]:
# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student1@esmt.edu')

# Keep True for offline demo; set False when organizer-managed proxy is available.
MOCK_MODE = False

# Ensure gateway URL is available even when .env is not loaded in notebook runtime.
API_BASE_URL = os.getenv(
    'WORKSHOP_API_BASE_URL',
    'https://gemini-workshop-gateway-395622257429.europe-west4.run.app',
)
os.environ['WORKSHOP_API_BASE_URL'] = API_BASE_URL

# Direct model IDs (no aliases).
model_catalog = get_workshop_model_catalog()
BASELINE_MODEL = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
ADVANCED_MODEL = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')
COUNTRY_MODEL = BASELINE_MODEL

print('API_BASE_URL =', API_BASE_URL)
print('STUDENT_EMAIL =', STUDENT_EMAIL)
print('MOCK_MODE =', MOCK_MODE)
print('Model choices =', model_catalog)
print('Selected baseline model =', BASELINE_MODEL)
print('Selected advanced model =', ADVANCED_MODEL)

# Exposed generation controls available in every processing call.
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40
MAX_TOKENS = 512
MAX_WORKERS = 8


## Step 6: Quick API Usage Demo (Single and Batch)


In [ ]:
# Raw LLM call API (single prompt) plus separate parser utilities.
single_prompt = '''Extract structured fields from this address and return JSON only:
1600 Pennsylvania Ave NW, Washington, DC 20500, USA
'''

try:
    raw_text = call_llm(
        single_prompt,
        email=STUDENT_EMAIL,
        model=BASELINE_MODEL,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        top_k=TOP_K,
        max_tokens=MAX_TOKENS,
        mock_mode=MOCK_MODE,
    )
    print('Raw LLM output:', raw_text)
    print('Parsed JSON:', parse_llm_json(raw_text))
    print('Parsed structured fields:', parse_llm_structured_address(raw_text))
except Exception as exc:
    print('Direct call failed:', exc)
    print('Check that WORKSHOP_EMAIL is a registered student email and gateway access is enabled.')

# Higher-level address processing wrappers remain available for stage pipelines.
batch_result = process_batch_addresses(
    [
        '1600 Pennsylvania Ave NW, Washington, DC 20500, USA',
        'Royal Opera House, Bow St, Covent Garden, London WC2E 9DD, United Kingdom',
    ],
    email=STUDENT_EMAIL,
    stage='baseline',
    model=BASELINE_MODEL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    max_workers=MAX_WORKERS,
    mock_mode=MOCK_MODE,
)
print('Batch result shape:', batch_result.shape)
batch_result.head(2)


## Step 7: Parameter Playground for Quality, Runtime, and Repeatability


In [ ]:
import time

# Use a small development subset for quick interactive experiments.
SAMPLE_SIZE = 20
REPEATS_PER_SETTING = 3

dev_sample = dev_df.head(SAMPLE_SIZE).copy()

# Modify this grid to explore behavior.
parameter_grid = [
    {'name': 'deterministic', 'temperature': 0.0, 'top_p': 1.0, 'top_k': 40},
    {'name': 'mild_sampling', 'temperature': 0.2, 'top_p': 0.95, 'top_k': 40},
    {'name': 'high_sampling', 'temperature': 0.7, 'top_p': 0.9, 'top_k': 80},
]


def run_once(temp: float, top_p: float, top_k: int):
    start = time.perf_counter()
    pred = process_batch_addresses(
        dev_sample,
        email=STUDENT_EMAIL,
        stage='baseline',
        model=BASELINE_MODEL,
        temperature=temp,
        top_p=top_p,
        top_k=top_k,
        max_tokens=MAX_TOKENS,
        max_workers=MAX_WORKERS,
        mock_mode=MOCK_MODE,
    )
    runtime_sec = time.perf_counter() - start
    report = evaluate_predictions(pred, dev_sample)
    return pred, report['summary']['micro_accuracy'], runtime_sec


def stability_vs_reference(reference_df: pd.DataFrame, run_df: pd.DataFrame) -> float:
    # Compare exact matches across scored fields, row by row.
    scored_fields = ['Town Name', 'Postal Code', 'Country Code (2 characters)']
    left = reference_df[['record_id', *scored_fields]].copy()
    right = run_df[['record_id', *scored_fields]].copy()

    merged = left.merge(right, on='record_id', suffixes=('_ref', '_run'))
    total = len(merged) * len(scored_fields)
    if total == 0:
        return 1.0

    matches = 0
    for field in scored_fields:
        ref = merged[f'{field}_ref'].fillna('').astype(str).str.strip().str.lower()
        cur = merged[f'{field}_run'].fillna('').astype(str).str.strip().str.lower()
        matches += int((ref == cur).sum())

    return matches / total


rows = []
for setting in parameter_grid:
    run_predictions = []
    run_scores = []
    run_times = []

    for run_idx in range(1, REPEATS_PER_SETTING + 1):
        pred_df, micro_acc, runtime_sec = run_once(
            setting['temperature'],
            setting['top_p'],
            setting['top_k'],
        )
        run_predictions.append(pred_df)
        run_scores.append(micro_acc)
        run_times.append(runtime_sec)

        rows.append(
            {
                'setting': setting['name'],
                'run': run_idx,
                'temperature': setting['temperature'],
                'top_p': setting['top_p'],
                'top_k': setting['top_k'],
                'micro_accuracy': round(float(micro_acc), 4),
                'runtime_sec': round(float(runtime_sec), 3),
                'stability_vs_run1': 1.0 if run_idx == 1 else round(stability_vs_reference(run_predictions[0], pred_df), 4),
            }
        )

    rows.append(
        {
            'setting': setting['name'],
            'run': 'aggregate',
            'temperature': setting['temperature'],
            'top_p': setting['top_p'],
            'top_k': setting['top_k'],
            'micro_accuracy': round(float(pd.Series(run_scores).mean()), 4),
            'runtime_sec': round(float(pd.Series(run_times).mean()), 3),
            'stability_vs_run1': round(
                float(pd.Series([stability_vs_reference(run_predictions[0], p) for p in run_predictions]).mean()),
                4,
            ),
        }
    )

playground_df = pd.DataFrame(rows)
playground_df


## Interpretation Guide

- Higher `micro_accuracy` means better field extraction quality on the sampled dev subset.
- Lower `runtime_sec` means faster batch processing.
- `stability_vs_run1` closer to `1.0` means higher repeatability.

After choosing your default parameters, keep them visible in every stage notebook.


## Next

Open `01_baseline_fast_model.ipynb` to run the first benchmark.


---

## Optional: Organizer-only student registration (reference)


# Admin: Student Registration Notebook

This notebook is for workshop organizers to create/update student accounts in the gateway.

Scope:
- bulk register students via `/register`,
- verify each student via `/user/{email}`,
- optional delete via `/user/{email}`,
- save operation logs to `outputs/admin`.

Important:
- `/register` must be enabled on the server (`APP_ALLOW_REGISTRATION_ENDPOINT=true`).
- Caller must have admin access to the gateway.


## Step 1: Install Dependencies


In [ ]:
%pip install -q requests pandas


## Step 2: Load Utilities and Resolve Project Root


In [ ]:
import os
import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/ESMT_Workshop')]:
    if (candidate / 'notebooks').exists() and (candidate / 'data').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Project root not found. Run this notebook from ESMT_Workshop repository.')

print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Gateway Configuration


In [ ]:
# Gateway URL from your current workshop environment.
SERVICE_URL = os.getenv(
    'WORKSHOP_API_BASE_URL',
    'https://gemini-workshop-gateway-395622257429.europe-west4.run.app',
).rstrip('/')

# Admin identity used by protected endpoints: /register and /user/{email}.
# The gateway accepts X-Admin-Email and can also accept Bearer auth.
ADMIN_EMAIL = os.getenv('WORKSHOP_ADMIN_EMAIL', 'btc.esmt.workshop@gmail.com').strip()
ADMIN_BEARER_TOKEN = os.getenv('WORKSHOP_ADMIN_BEARER_TOKEN', '').strip()

# Safety toggle:
# - True  -> build payload and show preview only
# - False -> execute real API calls
DRY_RUN = False

if not ADMIN_EMAIL and not ADMIN_BEARER_TOKEN:
    raise ValueError('Set WORKSHOP_ADMIN_EMAIL or WORKSHOP_ADMIN_BEARER_TOKEN for admin endpoints.')

print('SERVICE_URL =', SERVICE_URL)
print('ADMIN_EMAIL =', ADMIN_EMAIL)
print('ADMIN token provided =', bool(ADMIN_BEARER_TOKEN))
print('DRY_RUN =', DRY_RUN)


## Step 4: API Helper Functions


In [ ]:
def _headers() -> dict:
    headers = {'Content-Type': 'application/json'}
    if ADMIN_EMAIL:
        headers['X-Admin-Email'] = ADMIN_EMAIL
    if ADMIN_BEARER_TOKEN:
        headers['Authorization'] = f'Bearer {ADMIN_BEARER_TOKEN}'
    return headers


def _response_body(resp: requests.Response):
    try:
        return resp.json()
    except Exception:
        return resp.text


def health_check() -> dict:
    url = f"{SERVICE_URL}/health"
    resp = requests.get(url, timeout=20)
    return {
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def probe_admin_access() -> dict:
    """Non-destructive admin auth check for /register."""
    url = f"{SERVICE_URL}/register"
    payload = {'users': []}
    if DRY_RUN:
        return {
            'dry_run': True,
            'url': url,
            'payload_preview': payload,
        }

    resp = requests.post(url, headers=_headers(), json=payload, timeout=30)
    return {
        'dry_run': False,
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def register_students(users: list[dict]) -> dict:
    url = f"{SERVICE_URL}/register"
    payload = {'users': users}
    if DRY_RUN:
        return {
            'dry_run': True,
            'url': url,
            'payload_preview': payload,
        }

    resp = requests.post(url, headers=_headers(), json=payload, timeout=40)
    return {
        'dry_run': False,
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def get_student(email: str) -> dict:
    url = f"{SERVICE_URL}/user/{email}"
    resp = requests.get(url, headers=_headers(), timeout=20)
    return {
        'email': email,
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def delete_student(email: str) -> dict:
    url = f"{SERVICE_URL}/user/{email}"
    if DRY_RUN:
        return {
            'dry_run': True,
            'email': email,
            'url': url,
        }

    resp = requests.delete(url, headers=_headers(), timeout=20)
    return {
        'dry_run': False,
        'email': email,
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def run_full_registration_pipeline(users: list[dict]) -> tuple[dict, pd.DataFrame]:
    """Run register + verify in one call for organizer convenience."""
    register_result = register_students(users)

    verification_rows = []
    for email in [u['email'] for u in users]:
        response = get_student(email)
        row = {
            'email': email,
            'status_code': response['status_code'],
            'body': response['body'],
            'blocked': '',
            'requests_used': '',
            'request_limit': '',
            'concurrency_cap': '',
        }
        if isinstance(response['body'], dict):
            row['blocked'] = response['body'].get('blocked', '')
            row['requests_used'] = response['body'].get('requests_used', '')
            row['request_limit'] = response['body'].get('request_limit', '')
            row['concurrency_cap'] = response['body'].get('concurrency_cap', '')
        verification_rows.append(row)

    return register_result, pd.DataFrame(verification_rows)


## Step 5: Run Health and Admin Access Checks


In [ ]:
health = health_check()
admin_probe = probe_admin_access()

print('Health:', health)
print('Admin probe:', admin_probe)

if health['status_code'] != 200:
    raise RuntimeError(f'Health check failed: {health}')

if admin_probe['status_code'] != 200:
    raise RuntimeError(
        'Admin auth failed. Check WORKSHOP_ADMIN_EMAIL or WORKSHOP_ADMIN_BEARER_TOKEN. '
        f'Probe response: {admin_probe}'
    )


## Step 6: Define Student Input (Inline or CSV)


In [ ]:
# Option A: inline table (edit directly).
students_df = pd.DataFrame([
    {
        'email': 'student1@esmt.edu',
        'alias': 'Student 1',
        'request_limit': 15000,
        'concurrency_cap': 1,
    },
    {
        'email': 'student2@esmt.edu',
        'alias': 'Student 2',
        'request_limit': 15000,
        'concurrency_cap': 1,
    },
])

# Option B: load from CSV (uncomment and set path).
# csv_path = PROJECT_ROOT / 'data/input/student_emails.csv'
# students_df = pd.read_csv(csv_path)

students_df


## Step 7: Validate and Build `/register` Payload


In [ ]:
required_cols = ['email']
missing = [c for c in required_cols if c not in students_df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Normalize values and build API payload.
def _optional_int(value):
    if pd.isna(value) or str(value).strip() == '':
        return None
    return int(value)

users_payload = []
for _, row in students_df.fillna('').iterrows():
    item = {
        'email': str(row['email']).strip(),
    }

    alias = str(row.get('alias', '')).strip()
    if alias:
        item['alias'] = alias

    req_limit = _optional_int(row.get('request_limit', ''))
    if req_limit is not None:
        item['request_limit'] = req_limit

    conc_cap = _optional_int(row.get('concurrency_cap', ''))
    if conc_cap is not None:
        item['concurrency_cap'] = conc_cap

    users_payload.append(item)

print('Prepared users:', len(users_payload))
users_payload[:3]


## Step 8: Run Full Registration Pipeline


In [ ]:
register_result, verification_df = run_full_registration_pipeline(users_payload)
register_result


## Step 9: Verify Users with `/user/{email}`


In [ ]:
verification_df


## Step 10: Save Registration Log Artifacts


In [ ]:
save_dir = PROJECT_ROOT / 'outputs' / 'admin'
save_dir.mkdir(parents=True, exist_ok=True)

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

input_path = save_dir / f'registration_input_{run_id}.csv'
verify_path = save_dir / f'registration_verification_{run_id}.csv'
meta_path = save_dir / f'registration_meta_{run_id}.json'

students_df.to_csv(input_path, index=False)
verification_df.to_csv(verify_path, index=False)

meta = {
    'run_id': run_id,
    'service_url': SERVICE_URL,
    'admin_email': ADMIN_EMAIL,
    'dry_run': DRY_RUN,
    'register_result': register_result,
}
meta_path.write_text(json.dumps(meta, ensure_ascii=True, indent=2) + '')

print('Saved input:', input_path)
print('Saved verification:', verify_path)
print('Saved meta:', meta_path)


## Step 11 (Optional): Delete Students


In [ ]:
# Safety switch for delete operation.
ENABLE_DELETE = False

# By default uses the same emails as registration payload.
emails_to_delete = [u['email'] for u in users_payload]

if ENABLE_DELETE:
    delete_results = [delete_student(email) for email in emails_to_delete]
    pd.DataFrame(delete_results)
else:
    print('Deletion disabled. Set ENABLE_DELETE=True to execute deletes.')
